# 05 — Gold queries (samples)

Use these as starting points; pin to Power BI via the SQL analytics endpoint of `ObservabilityLH`.

In [1]:
import json, os
from pyspark.sql import functions as F

# When run inside Fabric, the notebook resource folder contains config.json.
# Fabric exposes notebook-attached files via mssparkutils / notebookutils.
try:
    import notebookutils  # type: ignore
    cfg_path = notebookutils.nbResPath + '/builtin/config.json'
    if not os.path.exists(cfg_path):
        # Fallback: lakehouse Files/config.json
        cfg_path = '/lakehouse/default/Files/config.json'
except Exception:
    cfg_path = './config.json'

with open(cfg_path, 'r', encoding='utf-8') as f:
    CFG = json.load(f)

OBS_WS  = CFG['observability_workspace_name']
OBS_LH  = CFG['observability_lakehouse_name']
TBL     = CFG['tables']
API     = CFG['fabric_api']
# monitored_workspaces is a list of workspace display names (strings).
# Backwards-compat: also accept the old [{workspace_name: ...}] shape.
_raw_mon = CFG['monitored_workspaces']
MONITOR = [m if isinstance(m, str) else m['workspace_name'] for m in _raw_mon]
INGEST  = CFG['ingestion']
print(f'Observability workspace : {OBS_WS}')
print(f'Observability lakehouse : {OBS_LH}')
print(f'Monitored workspaces    : {MONITOR}')

StatementMeta(, a1b1aaba-a622-4ece-ae3c-f49414621787, 3, Finished, Available, Finished, False)

Observability workspace : WS_OnelakeObservability
Observability lakehouse : lh_OnelakeObservability
Monitored workspaces    : ['WS_SagarFabric01', 'WS_SagarFabric03']


## Top users copying shortcut-backed data in the last 24h

In [3]:
df = spark.sql(f'''
SELECT executingUPN, target_type,
       COUNT(*)              AS ops,
       SUM(COALESCE(bytes,0)) AS bytes_read,
       COLLECT_SET(originatingApp)        AS apps,
       COLLECT_SET(callerIPAddress)       AS ips,
       COLLECT_SET(shortcut_path_consumer) AS shortcut_paths
FROM {TBL['silver']}
WHERE accessStartTime > current_timestamp() - INTERVAL 1 DAY
  AND operationCategory = 'Read'
GROUP BY executingUPN, target_type
ORDER BY ops DESC
''')

StatementMeta(, a1b1aaba-a622-4ece-ae3c-f49414621787, 5, Finished, Available, Finished, False)

In [4]:
display(df)

StatementMeta(, a1b1aaba-a622-4ece-ae3c-f49414621787, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0daf2da9-035e-4cde-aa11-629480ea65c6)

## Cross-workspace data egress (consumer ws ≠ target ws)

In [6]:
df1 = spark.sql(f'''
SELECT shortcut_consumer_workspace, target_workspace_id,
       executingUPN, originatingApp,
       COUNT(*) AS ops
FROM {TBL['silver']}
WHERE accessStartTime > current_timestamp() - INTERVAL 7 DAY
  AND target_type = 'OneLake'
  AND shortcut_consumer_workspace IS NOT NULL
GROUP BY shortcut_consumer_workspace, target_workspace_id, executingUPN, originatingApp
ORDER BY ops DESC
''')
display(df1)

StatementMeta(, a1b1aaba-a622-4ece-ae3c-f49414621787, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5fc54656-bf0c-4be1-986d-4785460d5b22)

## Suspicious external copies (PutBlobFromURL / CopyBlob via shortcut)

In [7]:
df2 = spark.sql(f'''
SELECT accessStartTime, executingUPN, callerIPAddress, originatingApp,
       operationName, shortcut_path_consumer, resolved_target_resource,
       target_type, target_location
FROM {TBL['silver']}
WHERE operationName IN ('PutBlobFromURL','CopyBlob','AbortCopyBlob')
  AND accessStartTime > current_timestamp() - INTERVAL 7 DAY
ORDER BY accessStartTime DESC
''')
display(df2)

StatementMeta(, a1b1aaba-a622-4ece-ae3c-f49414621787, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78da72a4-7d34-402f-92da-d8467de12007)

## External-tool access (azcopy, Storage Explorer, …) on any shortcut

In [ ]:
spark.sql(f'''
SELECT accessStartTime, executingUPN, callerIPAddress, originatingApp,
       operationName, shortcut_path_consumer, resolved_target_resource
FROM {TBL['silver']}
WHERE accessStartTime > current_timestamp() - INTERVAL 1 DAY
  AND originatingApp RLIKE '(?i)(azcopy|storage.explorer|powershell|curl|python-requests|rclone)'
ORDER BY accessStartTime DESC
''').show(100, truncate=False)